<a href="https://colab.research.google.com/github/dee431/Restraunt-Recommendation-Model-/blob/main/Restraunt_Recommendation_Model%F0%9F%8C%AD%F0%9F%8D%A9%F0%9F%8D%95%F0%9F%8D%A9%F0%9F%8C%AE%F0%9F%8D%94%E2%9D%84%EF%B8%8F%F0%9F%A5%AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Import Libraries**

In [1]:
# ==============================
# Cell 1: Import Libraries
# ==============================
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

**Cell 2: Upload and Load the Dataset**

In [2]:
# ==============================
# Cell 2: Load the Dataset
# ==============================
# Upload the file to Colab (or adjust path if already uploaded)
from google.colab import files
uploaded = files.upload()  # upload the CSV file

# Read the CSV (handle encoding and potential BOM)
df = pd.read_csv('Dataset .csv', encoding='utf-8-sig')
df.head()

Saving Dataset .csv to Dataset  (1).csv


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


**Cell 3: Data Exploration**

In [3]:
# ==============================
# Cell 3: Data Exploration
# ==============================
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nInfo:")
df.info()
print("\nMissing values:")
df.isnull().sum()
print("\nDescriptive stats:")
df.describe(include='all')

Dataset shape: (9551, 21)

Columns: ['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
count,9.551000e+03,9551,9551.000000,9551,9551,9551,9551,9551.000000,9551.000000,9542,...,9551,9551,9551,9551,9551,9551.000000,9551.000000,9551,9551,9551.000000
unique,NaN,7446,NaN,141,8918,1208,1265,NaN,NaN,1825,...,12,2,2,2,1,NaN,NaN,6,6,NaN
top,NaN,Cafe Coffee Day,NaN,New Delhi,"Dilli Haat, INA, New Delhi",Connaught Place,"Connaught Place, New Delhi",NaN,NaN,North Indian,...,Indian Rupees(Rs.),No,No,No,No,NaN,NaN,Orange,Average,NaN
freq,NaN,83,NaN,5473,11,122,122,NaN,NaN,936,...,8652,8393,7100,9517,9551,NaN,NaN,3737,3737,NaN
mean,9.051128e+06,NaN,18.365616,NaN,NaN,NaN,NaN,64.126574,25.854381,NaN,...,NaN,NaN,NaN,NaN,NaN,1.804837,2.666370,NaN,NaN,156.909748
std,8.791521e+06,NaN,56.750546,NaN,NaN,NaN,NaN,41.467058,11.007935,NaN,...,NaN,NaN,NaN,NaN,NaN,0.905609,1.516378,NaN,NaN,430.169145
min,5.300000e+01,NaN,1.000000,NaN,NaN,NaN,NaN,-157.948486,-41.330428,NaN,...,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,NaN,0.000000
25%,3.019625e+05,NaN,1.000000,NaN,NaN,NaN,NaN,77.081343,28.478713,NaN,...,NaN,NaN,NaN,NaN,NaN,1.000000,2.500000,NaN,NaN,5.000000
50%,6.004089e+06,NaN,1.000000,NaN,NaN,NaN,NaN,77.191964,28.570469,NaN,...,NaN,NaN,NaN,NaN,NaN,2.000000,3.200000,NaN,NaN,31.000000
75%,1.835229e+07,NaN,1.000000,NaN,NaN,NaN,NaN,77.282006,28.642758,NaN,...,NaN,NaN,NaN,NaN,NaN,2.000000,3.700000,NaN,NaN,131.000000


**Cell 4: Data Cleaning**

In [4]:
# ==============================
# Cell 4: Data Cleaning
# ==============================
# Work with a subset of relevant columns
# Fill missing Cuisines with empty string
df['Cuisines'] = df['Cuisines'].fillna('')

# Fill missing Average Cost for two with median
median_cost = df['Average Cost for two'].median()
df['Average Cost for two'] = df['Average Cost for two'].fillna(median_cost)

# Fill missing Price range with mode (if any)
if df['Price range'].isnull().any():
    df['Price range'] = df['Price range'].fillna(df['Price range'].mode()[0])

# Fill missing Aggregate rating with 0 (or median)
df['Aggregate rating'] = df['Aggregate rating'].fillna(0)

# Fill missing Votes with 0 (or median)
df['Votes'] = df['Votes'].fillna(0)

# Ensure numeric types
df['Average Cost for two'] = pd.to_numeric(df['Average Cost for two'], errors='coerce')
df['Average Cost for two'] = df['Average Cost for two'].fillna(median_cost)

# Check missing values after cleaning
print("Missing values after cleaning:")
print(df[['Cuisines', 'Average Cost for two', 'Price range', 'Aggregate rating']].isnull().sum())

Missing values after cleaning:
Cuisines                0
Average Cost for two    0
Price range             0
Aggregate rating        0
dtype: int64


**Cell 5: Feature Engineering (Multi‑hot Cuisines + Scaling)**

In [5]:
# ==============================
# Cell 5: Feature Engineering
# ==============================
# 1. Cuisines: split by comma and create multi-hot encoding
df['Cuisines_list'] = df['Cuisines'].apply(lambda x: [c.strip() for c in x.split(',') if c.strip() != ''])

mlb = MultiLabelBinarizer()
cuisine_matrix = mlb.fit_transform(df['Cuisines_list'])
cuisine_df = pd.DataFrame(cuisine_matrix, columns=mlb.classes_)

# 2. Numerical features: Price range, Average Cost for two, Aggregate rating
numerical = df[['Price range', 'Average Cost for two', 'Aggregate rating']].copy()
scaler = StandardScaler()
numerical_scaled = scaler.fit_transform(numerical)
numerical_df = pd.DataFrame(numerical_scaled, columns=['Price_range_scaled', 'Avg_cost_scaled', 'Rating_scaled'])

# 3. Combine all features into a single matrix
features = pd.concat([cuisine_df, numerical_df], axis=1)
feature_matrix = features.values

**Cell 6: Recommendation Function (Content‑Based)**

In [6]:
# ==============================
# Cell 6: Recommendation Function
# ==============================
def recommend_restaurants(user_prefs, df, mlb, scaler, feature_matrix, top_n=10):
    """
    user_prefs: dict with keys:
        - 'cuisines': list of preferred cuisines (strings)
        - 'city': preferred city (optional)
        - 'price_range': preferred price range (1-4, optional)
        - 'max_cost': maximum average cost for two (optional)
        - 'min_rating': minimum rating threshold (optional)
    Returns a DataFrame with top_n recommended restaurants.
    """
    # Filter by city if provided
    filtered = df.copy()
    if user_prefs.get('city'):
        filtered = filtered[filtered['City'].str.lower() == user_prefs['city'].lower()]

    if filtered.empty:
        return pd.DataFrame()

    # Filter by price range if provided
    if user_prefs.get('price_range'):
        filtered = filtered[filtered['Price range'] == user_prefs['price_range']]

    # Filter by max cost if provided
    if user_prefs.get('max_cost'):
        filtered = filtered[filtered['Average Cost for two'] <= user_prefs['max_cost']]

    # Filter by min rating if provided
    if user_prefs.get('min_rating'):
        filtered = filtered[filtered['Aggregate rating'] >= user_prefs['min_rating']]

    if filtered.empty:
        return pd.DataFrame()

    # Build user vector
    # Cuisine vector: 1 for preferred cuisines, 0 otherwise
    user_cuisine = np.zeros(len(mlb.classes_))
    preferred = [c.strip() for c in user_prefs.get('cuisines', [])]
    for c in preferred:
        if c in mlb.classes_:
            idx = list(mlb.classes_).index(c)
            user_cuisine[idx] = 1

    # Numerical vector: use user's specified values, fallback to mean of filtered
    user_num = np.array([
        user_prefs.get('price_range', filtered['Price range'].mean()),
        user_prefs.get('max_cost', filtered['Average Cost for two'].mean()),
        user_prefs.get('min_rating', filtered['Aggregate rating'].mean())
    ])
    user_num_scaled = scaler.transform(user_num.reshape(1, -1)).flatten()

    user_vector = np.concatenate([user_cuisine, user_num_scaled])

    # Get indices of filtered restaurants
    indices = filtered.index.tolist()
    filtered_feature_matrix = feature_matrix[indices]

    # Compute cosine similarity between user vector and all filtered restaurants
    similarities = cosine_similarity([user_vector], filtered_feature_matrix)[0]

    # Get top N indices
    top_indices = np.argsort(similarities)[::-1][:top_n]

    # Return recommendations
    recs = filtered.iloc[top_indices]
    recs = recs[['Restaurant ID', 'Restaurant Name', 'City', 'Cuisines',
                 'Average Cost for two', 'Price range', 'Aggregate rating', 'Votes']]
    recs['Similarity Score'] = similarities[top_indices]
    return recs

**Cell 7: Test the Recommendation System**

In [7]:
# ==============================
# Cell 7: Test the Recommendation System
# ==============================
# Example 1: Italian/Pizza in New Delhi, price range 3, max cost 1500, min rating 4.0
user_prefs = {
    'cuisines': ['Italian', 'Pizza'],
    'city': 'New Delhi',
    'price_range': 3,
    'max_cost': 1500,
    'min_rating': 4.0
}

recommendations = recommend_restaurants(user_prefs, df, mlb, scaler, feature_matrix, top_n=10)
print("Top 10 recommendations:")
print(recommendations)

# Example 2: Japanese/Sushi in Bangalore, min rating 4.5
user_prefs2 = {
    'cuisines': ['Japanese', 'Sushi'],
    'city': 'Bangalore',
    'min_rating': 4.5
}
rec2 = recommend_restaurants(user_prefs2, df, mlb, scaler, feature_matrix, top_n=5)
print("\nTop 5 Japanese restaurants in Bangalore:")
print(rec2)

Top 10 recommendations:
      Restaurant ID             Restaurant Name       City  \
3658       18400736                 Owl is Well  New Delhi   
6461       18219554             The Coffee Shop  New Delhi   
3702       18386761              Roadhouse Cafe  New Delhi   
3696         310776  Gastronomica Kitchen & Bar  New Delhi   
3770       18435305          Chhalava - �__Lava  New Delhi   
3258       18415346                   Cafe Yell  New Delhi   
7196       18291469                  Puppychino  New Delhi   
2635         307113                      Diggin  New Delhi   
6717       18254514             The Drunk House  New Delhi   
6701         306476            AMPM Caf�� & Bar  New Delhi   

                                               Cuisines  Average Cost for two  \
3658        Burger, American, Fast Food, Italian, Pizza                  1000   
6461                                      Cafe, Italian                  1200   
3702                               Continental, It

**Cell 8: Find Similar Restaurants by ID (Optional)**

In [8]:
# ==============================
# Cell 8: Recommend Similar Restaurants (by Restaurant ID)
# ==============================
def recommend_similar_restaurants(restaurant_id, df, feature_matrix, top_n=10):
    """
    Given a restaurant ID, find the most similar restaurants based on features.
    """
    idx = df[df['Restaurant ID'] == restaurant_id].index
    if len(idx) == 0:
        return pd.DataFrame()
    idx = idx[0]
    restaurant_vector = feature_matrix[idx].reshape(1, -1)
    similarities = cosine_similarity(restaurant_vector, feature_matrix)[0]
    # Exclude the restaurant itself
    similar_indices = np.argsort(similarities)[::-1][1:top_n+1]
    recs = df.iloc[similar_indices][['Restaurant ID', 'Restaurant Name', 'City', 'Cuisines',
                                     'Average Cost for two', 'Price range', 'Aggregate rating', 'Votes']]
    recs['Similarity Score'] = similarities[similar_indices]
    return recs

# Test: find restaurants similar to "Le Petit Souffle" (ID: 6317637)
similar = recommend_similar_restaurants(6317637, df, feature_matrix, top_n=5)
print("\nRestaurants similar to Le Petit Souffle:")
print(similar)


Restaurants similar to Le Petit Souffle:
      Restaurant ID                 Restaurant Name         City  Cuisines  \
9299        7001086                           Milse     Auckland  Desserts   
1           6304287                Izakaya Kikufuji  Makati City  Japanese   
259        17259340                          Django   Des Moines    French   
1466         309125                         Kuuraku      Gurgaon  Japanese   
9484          75989  Restaurant Mosaic @ The Orient     Pretoria    French   

      Average Cost for two  Price range  Aggregate rating  Votes  \
9299                    50            3               4.9    754   
1                     1200            3               4.5    591   
259                     40            3               4.3    532   
1466                  1250            3               3.9    106   
9484                  3210            4               4.9     85   

      Similarity Score  
9299          0.837557  
1             0.835861  
259  

**Cell 9: Discussion and Evaluation (Optional)**

In [9]:
# ==============================
# Cell 9: Evaluation / Discussion
# ==============================
# The system is content‑based and can be extended with location, popularity, etc.
# Users can manually inspect the quality of recommendations.
print("\nRecommendation system ready!")


Recommendation system ready!


**Cell 10: Install Additional Libraries (if needed)**

In [10]:
# ==============================
# Cell 10: Install ipywidgets and Plotly
# ==============================
!pip install ipywidgets plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.7 MB/s eta 0:00:00


**Cell 11: Import Dashboard Libraries**

In [11]:
# ==============================
# Cell 11: Import Libraries for Dashboard
# ==============================
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px
import plotly.graph_objects as go

**Cell 12: Define the Recommendation Function (Wrapper)**



In [12]:
# ==============================
# Cell 12: Recommendation Function (already defined in Cell 6)
# ==============================
# If you haven't run Cell 6, copy the function definition here.
# We'll assume it's already defined.

**Cell 13: Build the Interactive Dashboard UI**

In [16]:
# ==============================
# Cell 13: Create Dashboard Widgets
# ==============================
# Get unique cities from dataset
cities = sorted(df['City'].dropna().unique())
city_dropdown = widgets.Dropdown(
    options=cities,
    value=cities[0] if len(cities) > 0 else None,
    description='City:',
    layout=widgets.Layout(width='300px')
)

# Get all cuisines from the MultiLabelBinarizer classes
all_cuisines = sorted(mlb.classes_)
cuisine_select = widgets.SelectMultiple(
    options=all_cuisines,
    value=[],
    description='Cuisines:',
    layout=widgets.Layout(width='400px', height='200px')
)

# Price range slider (1-4)
price_slider = widgets.IntSlider(
    value=2, min=1, max=4, step=1,
    description='Price range:',
    layout=widgets.Layout(width='300px')
)

# Max cost slider (in multiples of 500, up to 10000)
max_cost_slider = widgets.IntSlider(
    value=2000, min=200, max=10000, step=200,
    description='Max cost:',
    layout=widgets.Layout(width='300px')
)

# Min rating slider (0-5)
rating_slider = widgets.FloatSlider(
    value=4.0, min=0, max=5, step=0.1,
    description='Min rating:',
    layout=widgets.Layout(width='300px')
)

# Number of recommendations
top_n_slider = widgets.IntSlider(
    value=10, min=5, max=25, step=5,
    description='Top N:',
    layout=widgets.Layout(width='300px')
)

# Button to trigger recommendation
recommend_btn = widgets.Button(
    description='Get Recommendations',
    button_style='success',
    icon='search'
)

# Output areas
output_table = widgets.Output()
output_charts = widgets.Output()

# Layout the widgets
ui = widgets.VBox([
    widgets.HBox([city_dropdown, cuisine_select]),
    widgets.HBox([price_slider, max_cost_slider, rating_slider]),
    widgets.HBox([top_n_slider, recommend_btn]),
    widgets.HTML("<hr>"),
    widgets.HBox([output_table, output_charts])
])

display(ui)

**Cell 14: Define the Callback for the Button**



In [18]:
# ==============================
# Cell 14: Recommendation Callback
# ==============================
def on_recommend_clicked(b):
    # Clear previous outputs
    output_table.clear_output()
    output_charts.clear_output()

    # Collect user preferences
    user_prefs = {
        'cuisines': list(cuisine_select.value),
        'city': city_dropdown.value,
        'price_range': price_slider.value,
        'max_cost': max_cost_slider.value,
        'min_rating': rating_slider.value
    }

    # Get recommendations
    recs = recommend_restaurants(user_prefs, df, mlb, scaler, feature_matrix, top_n=top_n_slider.value)

    if recs.empty:
        with output_table:
            print("No restaurants found matching your criteria.")
        return

    # Display recommendations table
    with output_table:
        display(recs[['Restaurant Name', 'City', 'Cuisines', 'Average Cost for two',
                      'Price range', 'Aggregate rating', 'Votes', 'Similarity Score']])

    # Create visualizations with Plotly
    with output_charts:
        # 1. Bar chart of ratings
        fig_rating = px.bar(
            recs,
            x='Restaurant Name',
            y='Aggregate rating',
            color='Similarity Score',
            color_continuous_scale='Blues',
            title='Top Recommendations by Rating',
            labels={'Aggregate rating': 'Rating', 'Restaurant Name': ''}
        )
        fig_rating.update_layout(showlegend=False, xaxis_tickangle=-45)
        fig_rating.show()

        # 2. Cuisine distribution (from the recommended restaurants)
        # Collect all cuisines from the recommendations
        cuisine_counts = {}
        for cuisines in recs['Cuisines']:
            for c in cuisines.split(','):
                c = c.strip()
                cuisine_counts[c] = cuisine_counts.get(c, 0) + 1
        if cuisine_counts:
            pie_data = pd.DataFrame({
                'Cuisine': list(cuisine_counts.keys()),
                'Count': list(cuisine_counts.values())
            }).sort_values('Count', ascending=False)
            fig_pie = px.pie(
                pie_data,
                names='Cuisine',
                values='Count',
                title='Cuisine Distribution Among Recommendations',
                hole=0.4
            )
            fig_pie.show()
        else:
            print("No cuisine data to visualize.")

# Attach callback to button
recommend_btn.on_click(on_recommend_clicked)